In [1]:
import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.append(os.path.abspath('..'))

# Set styling for plots
sns.set_theme(style="whitegrid")

# End-to-End Evaluation of a Safety-Aware Adaptive Pricing Engine

This notebook evaluates the pricing engine as a complete decision-making system
prior to any performance benchmarking or deployment.

The objective is not to maximize headline metrics, but to verify that the system
is statistically sound, behaviorally safe, and robust to realistic failure modes.

Evaluation is conducted offline and in simulated online settings, using strictly
out-of-sample data and explicit safety constraints. Only if all evaluation criteria
are satisfied does the system proceed to benchmarking.



The pricing engine is composed of four independent subsystems:

• Demand estimation models that output booking probabilities  
• Adaptive pricing policies that select prices under uncertainty  
• A safety governor that enforces hard business and regulatory constraints  
• An evaluation layer that audits correctness, stability, and trustworthiness  

This notebook evaluates the interaction of these subsystems, rather than any
single model in isolation.


The evaluation follows three core principles:

1. Separation of concerns  
   Learning, safety, and evaluation are treated as independent systems.

2. Out-of-sample correctness  
   No model or policy is evaluated on data it was trained on.

3. Failure-first design  
   The evaluation prioritizes identifying unsafe or misleading behavior over
   reporting optimistic performance.


The output of this notebook is a binary decision:

• GO   — the pricing engine is correct and robust enough to be benchmarked  
• NO-GO — the system requires further modeling or safety iteration  

This decision is based on statistical validity, safety correctness, learning
behavior, and robustness under stress.


In [2]:
import numpy as np
import random

np.random.seed(42)
random.seed(42)

print("Evaluation environment initialized.")


Evaluation environment initialized.


We load the pricing engine as a unified module. All evaluation components operate
on this shared implementation to ensure consistency across experiments.


In [3]:
import pricing_engine as pe

print("Pricing engine loaded.")


Pricing engine loaded.


## Data Integrity, Feature Contract & Temporal Split

The evaluation operates on historical booking data prepared by the data loading
pipeline. During loading, prices for booked nights are recovered using
listing-level temporal imputation to ensure that all realized bookings have valid
prices.

This preprocessing step is essential for counterfactual revenue estimation and
is treated as part of data engineering rather than evaluation. The evaluation
therefore validates assumptions on the prepared data, without reapplying or
replicating the cleaning logic.


In [4]:
from pathlib import Path



try:
    SCRIPT_DIR = Path(__file__).parent
except NameError:
    SCRIPT_DIR = Path.cwd()

# SCRIPT_DIR is the 'notebooks' folder.
# SCRIPT_DIR.parent is the 'Dynamic Pricing Engine' folder.
PROJECT_ROOT = SCRIPT_DIR.parent

# Now build the path from the project root
CALENDAR_PATH = PROJECT_ROOT / 'data' / 'calendar.csv'
LISTINGS_PATH = PROJECT_ROOT / 'data' / 'listings.csv'



In [5]:
df = pe.load_and_clean_seattle_data(CALENDAR_PATH, LISTINGS_PATH)
df.head()

,listing_id,date,available,price,is_booked,neighborhood
0,3335,2016-01-04,f,120.0,1,Rainier Valley
1,3335,2016-01-05,f,120.0,1,Rainier Valley
2,3335,2016-01-06,f,120.0,1,Rainier Valley
3,3335,2016-01-07,f,120.0,1,Rainier Valley
4,3335,2016-01-08,f,120.0,1,Rainier Valley


Before feature construction or model training, we verify that the prepared dataset
satisfies the minimal structural and domain assumptions required for offline
evaluation.


In [6]:
required_columns = {"price", "is_booked", "date", "neighborhood", "listing_id"}

assert required_columns.issubset(df.columns), "Required columns missing"
assert df["price"].min() > 0, "Non-positive prices detected"
assert set(df["is_booked"].unique()).issubset({0, 1}), "Invalid booking labels"

print("Prepared data passed structural and domain checks.")


Prepared data passed structural and domain checks.


Model features are derived explicitly from the prepared dataset to ensure
transparency, reproducibility, and consistency across all demand models and pricing
policies.

All feature transformations are applied prior to any model training or evaluation.


In [7]:
# Calendar-based features
df["day_of_week"] = df["date"].dt.weekday          # 0 = Monday
df["month"] = df["date"].dt.month

# Weekend indicator (Saturday = 5, Sunday = 6)
df["is_weekend"] = df["day_of_week"].isin([5, 6]).astype(int)

print("Temporal features constructed.")


Temporal features constructed.


Demand models require purely numeric inputs. Categorical variables are therefore
encoded explicitly at the evaluation level to ensure a stable and reproducible
feature interface.

Encoding is deterministic and frozen for the remainder of the evaluation.


In [8]:
# Encode neighborhood as numeric codes (deterministic)
df["neighborhood_code"] = (
    df["neighborhood"]
    .astype("category")
    .cat.codes
)

df[["neighborhood", "neighborhood_code"]].head()


,neighborhood,neighborhood_code
0,Rainier Valley,13
1,Rainier Valley,13
2,Rainier Valley,13
3,Rainier Valley,13
4,Rainier Valley,13


All demand models and pricing policies operate on the same explicit feature set.
This feature contract is frozen for the remainder of the evaluation to prevent
information leakage and ensure meaningful comparisons.


In [9]:
feature_cols = [
    "neighborhood_code",
    "day_of_week",
    "is_weekend",
    "month",
]

missing_features = set(feature_cols) - set(df.columns)
assert not missing_features, f"Missing features after engineering: {missing_features}"

print(f"Feature contract established with {len(feature_cols)} features:")
feature_cols


Feature contract established with 4 features:


['neighborhood_code', 'day_of_week', 'is_weekend', 'month']

Pricing systems operate in non-stationary environments. To reflect real deployment
conditions and avoid optimistic bias, we adopt a time-aware evaluation protocol.

Demand models are trained exclusively on historical data and evaluated only on
future observations. No model or policy is ever trained or tuned on evaluation data.


In [10]:
df = df.sort_values("date")

split_point = df["date"].quantile(0.8)

train_df = df[df["date"] <= split_point].copy()
eval_df  = df[df["date"] > split_point].copy()

print("Training period :", train_df["date"].min(), "→", train_df["date"].max())
print("Evaluation period:", eval_df["date"].min(),  "→", eval_df["date"].max())
print("Train rows:", len(train_df), "| Eval rows:", len(eval_df))


Training period : 2016-01-04 00:00:00 → 2016-10-21 00:00:00
Evaluation period: 2016-10-22 00:00:00 → 2017-01-02 00:00:00
Train rows: 1114856 | Eval rows: 278714


The evaluation dataset represents future market conditions relative to the training
data. All subsequent analysis—including calibration, counterfactual value
estimation, learning dynamics, and stress testing—operates exclusively on this
held-out set.


In [11]:
assert len(train_df) > 0 and len(eval_df) > 0, "Empty split detected"
assert train_df["date"].max() < eval_df["date"].min(), "Temporal leakage detected"

print("Temporal split validated and evaluation protocol locked.")


Temporal split validated and evaluation protocol locked.


## Demand Model Training & Probabilistic Trust Gate

All pricing decisions in the engine are driven by probabilistic estimates of booking
likelihood. Before evaluating any pricing policy, we must therefore establish that
these probability estimates are statistically trustworthy.

In this step, a single demand model is trained on historical data and evaluated
out-of-sample on future observations. This mirrors real deployment, where one
probability oracle is consumed by all pricing policies.


The pricing engine relies on a hierarchical Bayesian logistic demand model that
shares statistical strength across listings while allowing local adaptation.

Policy evaluation is therefore conditioned on this demand model alone. Alternative
demand models may be evaluated in separate runs, but probabilities are never mixed
within a single evaluation.


In [12]:
from pricing_engine import HierarchicalBayesianLogisticDemand

BETA_PRICE = -2.0  # example value; fixed a priori

demand_model = HierarchicalBayesianLogisticDemand(
    beta_price=BETA_PRICE,
    min_listing_obs=30
)

print("Hierarchical Bayesian demand model instantiated.")


Hierarchical Bayesian demand model instantiated.


The demand model is trained exclusively on the training dataset derived from
historical data. Once trained, model parameters are frozen and no retraining or
tuning occurs during evaluation.

This separation between training and evaluation is critical for honest assessment
of downstream pricing policies.


In [13]:
demand_model.fit(
    train_df,
    feature_cols
)

print(f"Demand model trained on {len(train_df)} observations.")


Demand model trained on 1114856 observations.


A demand model may achieve good average accuracy while still producing unreliable
probability estimates. Because pricing policies act directly on predicted booking
probabilities, calibration is evaluated explicitly.

We assess probabilistic correctness using the Brier score and Expected Calibration
Error (ECE). Poor calibration invalidates counterfactual policy evaluation and
constitutes a hard stop.


In [14]:
from pricing_engine.evaluation.offline.demand_calibration import DemandCalibrationEvaluator

calibrator = DemandCalibrationEvaluator(n_bins=10)

cal_result = calibrator.evaluate(
    demand_model,
    eval_df
)

print(
    f"{cal_result.model_name} | "
    f"Brier Score: {cal_result.brier_score:.4f} | "
    f"ECE: {cal_result.ece:.4f}"
)


HierarchicalBayesianLogisticDemand | Brier Score: 0.2021 | ECE: 0.0408


Only demand models that satisfy calibration requirements are eligible for policy
evaluation. If this gate fails, downstream counterfactual estimates are not
statistically meaningful.


In [15]:
MAX_ECE = 0.05

calibration_ok = cal_result.ece <= MAX_ECE

print("Calibration gate passed:", calibration_ok)


Calibration gate passed: True


## Safety-Aware Counterfactual Policy Value Estimation

We now evaluate adaptive pricing policies as they would operate in production.

Policies are stateful learning agents that:

• consume probabilistic demand estimates from a fixed demand model

• update online based on observed outcomes

• operate under strict safety constraints

Evaluation is performed via sequential replay of historical data, ensuring that
learning dynamics, safety behavior, and economic value are assessed jointly.


Bandit policies treat the demand model as a black-box prior that maps feature vectors
to booking probability estimates with uncertainty.

This interface ensures that policies remain model-agnostic and do not inspect
internal demand model parameters.


In [16]:
# ------------------------------------------------------------
# FAST OFFLINE PRIOR: Precompute demand at historical price
# ------------------------------------------------------------
eval_df = eval_df.copy()

eval_df["_p_hat"] = eval_df.apply(
    lambda r: demand_model.predict(r.to_dict(), r.price).prob,
    axis=1
)

print("Precomputed demand probabilities at historical prices.")


Precomputed demand probabilities at historical prices.


In [17]:
# ------------------------------------------------------------
# FAST PRIOR for OFFLINE evaluation (elasticity approximation)
# ------------------------------------------------------------
def prior_predict_fn_factory(base_prob, base_price):
    """
    Creates a fast local prior predictor using elasticity approximation.
    """

    def prior_predict_fn(X: np.ndarray):
        prices = np.expm1(X[:, -1])
        ratio = prices / base_price

        # Log-linear elasticity approximation
        probs = np.clip(
            base_prob * np.power(ratio, BETA_PRICE),
            0.01,
            0.99
        )

        # Conservative fixed uncertainty
        stds = np.full_like(probs, 0.05)

        return probs, stds

    return prior_predict_fn


Bandits reuse the same feature scaling as the demand model to ensure consistent
geometry between offline priors and online updates.


In [18]:
bandit_feature_names = feature_cols
bandit_scaler = demand_model.scaler


We evaluate multiple pricing policies under identical conditions:

• a historical baseline (no learning)

• Thompson Sampling

• Bayesian UCB

• LinUCB

All policies observe the same contexts and are subject to the same safety constraints.


In [20]:
from pricing_engine import (
    ThompsonPricingBandit,
    BayesianUCBBandit,
    LinUCBBandit,
    PricingDecision
)

# Historical baseline
class HistoricalPricingPolicy:
    name = "Historical"

    def choose_price(self, context, **kwargs):
        return PricingDecision(
            selected_price=context["price"],
            expected_revenue=context["price"],
            uncertainty_sigma=0.0,
            source="Historical",
            panic_mode=False,
        )

    def update(self, *args, **kwargs):
        pass  # no learning


policies = {
    "Thompson": ThompsonPricingBandit(
        feature_names=bandit_feature_names,
        scaler=bandit_scaler,
        predict_fn=prior_predict_fn_factory,
    ),
    "BayesianUCB": BayesianUCBBandit(
        feature_names=bandit_feature_names,
        scaler=bandit_scaler,
        predict_fn=prior_predict_fn_factory,
        beta=1.5,
    ),
    "LinUCB": LinUCBBandit(
        feature_names=bandit_feature_names,
        scaler=bandit_scaler,
        predict_fn=prior_predict_fn_factory,
        alpha=1.0,
    ),
    "Historical": HistoricalPricingPolicy(),
}

print("Policies instantiated:", list(policies.keys()))


Policies instantiated: ['Thompson', 'BayesianUCB', 'LinUCB', 'Historical']


Safety constraints are enforced exactly as they would be in production. Policy
outputs are validated and clamped before revenue estimation or learning updates.


In [21]:
from pricing_engine import SafetyGovernor, SafetyConfig

safety = SafetyGovernor(
    SafetyConfig(
        min_price_global=10.0,
        max_price_global=5000.0,
        min_margin_dollars=10.0,
        max_daily_change_pct=0.25,
        uncertainty_penalty_pct=0.15,
    )
)


Historical data is replayed sequentially. For each policy and timestep:

• the policy proposes a price

• demand uncertainty is computed

• the safety governor clamps and smooths the price

• counterfactual revenue is estimated

• the policy updates based on observed outcomes

All safety signals are recorded for downstream analysis.


In [ ]:
from collections import defaultdict
import time

dr_revenue = defaultdict(list)
safety_logs = defaultdict(list)

prev_prices = {name: None for name in policies}

t0 = time.perf_counter()

for idx, row in eval_df.iterrows():
    context = row.to_dict()
    p_hist = row["price"]
    y = row["is_booked"]
    p_hat = row["_p_hat"]

    constraints = {
        "min_price": 0.5 * p_hist,
        "max_price": 3.0 * p_hist,
        "cost_basis": 0.4 * p_hist,
    }

    # Create fast prior bound to this row
    prior_predict_fn = prior_predict_fn_factory(p_hat, p_hist)

    for name, policy in policies.items():

        # Inject fast prior (offline only)
        if hasattr(policy, "predict_fn"):
            policy.predict_fn = prior_predict_fn

        # 1. Policy proposal
        decision = policy.choose_price(
            context,
            min_p=constraints["min_price"],
            max_p=constraints["max_price"],
        )

        # 2. True demand prediction (pre-safety)
        pred_pre = demand_model.predict(context, decision.selected_price)

        # 3. Safety governor
        safety_res = safety.validate_and_clamp(
            suggested_price=decision.selected_price,
            constraints=constraints,
            prev_price=prev_prices[name],
            demand_meta={"std_dev": pred_pre.std_dev},
        )

        p_safe = safety_res.safe_price
        prev_prices[name] = p_safe

        # 4. True demand at safe price
        pred_safe = demand_model.predict(context, p_safe)
        exp_rev = p_safe * pred_safe.prob

        # 5. Doubly Robust correction
        if abs(p_safe - p_hist) < 10.0:
            obs_rev = p_hist * y
            exp_hist = p_hist * p_hat
            dr = exp_rev + (obs_rev - exp_hist)
        else:
            dr = exp_rev

        dr_revenue[name].append(dr)
        safety_logs[name].append(safety_res)

        # 6. Online update
        if hasattr(policy, "update"):
            policy.update(context, p_safe, y)

elapsed = time.perf_counter() - t0
print(f"Flow 4 evaluation completed in {elapsed:.2f} seconds.")
print(f"Throughput: {len(eval_df)/elapsed:.1f} rows/sec")


In [ ]:
for name, vals in dr_revenue.items():
    print(f"{name} | Estimated Revenue: {np.sum(vals):.2f}")
